In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zipf, entropy
import pandas as pd
import math

names_latency=['time_msec', 'latency', 'op2', 'write_size', 'op3']

def analyze_lba_distribution(lbas, bin_size=4096, zipf_s=1.2, n_zones=10):
    """
    Analyze LBA access pattern and compare to uniform, zipfian, and zoned.
    
    Args:
        lbas (array-like): list of logical block addresses
        bin_size (int): bin granularity (default 4KB)
        zipf_s (float): Zipf exponent (default 1.2, tweak as needed)
        n_zones (int): number of zones for zoned model
    """
    lbas = np.array(lbas)
    max_lba = lbas.max()
    
    # Bin LBAs
    bins = np.arange(0, max_lba + bin_size, bin_size)
    hist, _ = np.histogram(lbas, bins=bins)
    emp_probs = hist / hist.sum()
    
    # Filter out empty bins for fair comparison
    nonzero = emp_probs > 0
    emp_probs = emp_probs[nonzero]
    nbins = len(emp_probs)
    
    # --- Candidate distributions ---
    # Uniform
    uniform_probs = np.ones(nbins) / nbins
    
    # Zipf (rank-based)
    ranks = np.arange(1, nbins + 1)
    zipf_probs = zipf.pmf(ranks, a=zipf_s)
    zipf_probs /= zipf_probs.sum()
    
    # Zoned (hot zones get more traffic)
    zoned_probs = np.ones(nbins) * 0.01
    hot_bins = np.linspace(0, nbins-1, n_zones, dtype=int)
    zoned_probs[hot_bins] += 1
    zoned_probs /= zoned_probs.sum()
    
    # --- Similarity (KL divergence) ---
    kl_uniform = entropy(emp_probs, uniform_probs)
    kl_zipf = entropy(emp_probs, zipf_probs)
    kl_zoned = entropy(emp_probs, zoned_probs)
    
    results = {
        "Uniform": kl_uniform,
        "Zipfian": kl_zipf,
        "Zoned": kl_zoned
    }
    closest = min(results, key=results.get)
    
    print("KL Divergence scores (lower is better):")
    for k, v in results.items():
        print(f"  {k:8s}: {v:.4f}")
    print(f"\n👉 Closest match: {closest}")
    
    # --- Plot distributions ---
    plt.figure(figsize=(8, 5))
    plt.plot(emp_probs, label="Empirical", lw=2)
    plt.plot(uniform_probs, label="Uniform", ls="--")
    plt.plot(zipf_probs, label=f"Zipf(s={zipf_s})", ls="-.")
    plt.plot(zoned_probs, label="Zoned", ls=":")
    plt.yscale("log")
    plt.xlabel("Bin index (sorted)")
    plt.ylabel("Probability")
    plt.title("LBA Access Distribution")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return closest, results



In [ ]:
import re

def parse_lba_len_file(filename):
    lbas = []
    lens = []
    
    # regex for "lba: number" and "len: number"
    pattern = re.compile(r"lba:\s*(\d+).*?len:\s*(\d+)")
    
    with open(filename, "r") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                lbas.append(int(match.group(1)))
                lens.append(int(match.group(2)))
    
    # build DataFrame
    df = pd.DataFrame({
        "lba": lbas,
        "len": lens
    })
    
    return df


In [ ]:
lbas_len =['lbas', 'len']
lbas_len_df = parse_lba_len_file('/home/surbhi/measurements/zfs-resilvering-wb/io-pattern/dmesg_resilveringLBAS')
lbas = lbas_len_df['lbas'].to_numpy()
closest, results = analyze_lba_distribution(lbas)


ParserError: Error tokenizing data. C error: Expected 2 fields in line 26220, saw 3
